**Author**: Felipe Matheus  
**Start Date**: 12/06/2026  
**End Date**: 24/06/2026  
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TARGET = "iacs_final"
ALL_FEATURES = ["purity", "iacs", "temperature", "time"]
PROCESS = "annealing_iacs"

TAG = "annealing-v1-best-quality"
GRID = {
    "time_limit_a": [900, 1500],
    "num_bag_folds_a": [10, 20],
    "weight_on_essay_rows": [1.0, 2.0],
    # "features": [
    #     ("purity", "iacs", "temperature", "time"),
    #     ("iacs", "temperature", "time"),
    # ],
}

# 2. Data (same preparation as annealing_iacs.ipynb, run once)

In [3]:
SCHEMA_DATE = "080626"
FILE_NAME = "dataset_annealing_iacs.csv"
FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"
FILE_NAME_VALIDATION_DATA = "Experimental Results Annealing V2 - CompiledResults.csv"

# ---- Schema (essay) data: explicit is_essay marker ----
df_raw_schema = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
df_schema["is_essay"] = True

# ---- Literature data ----
df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
)
df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
df_lit["is_essay"] = False

# ---- Validation data ----
df_val = proc.load_validation_data_chimie_paris(
    path=os.path.join(varv.PATHS.data_processed, FILE_NAME_VALIDATION_DATA)
)

assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# ---- Concat. weight_col is built INSIDE the runr from is_essay + config ----
df = pd.concat([df_schema, df_lit], ignore_index=True)
print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
df.head()

Dataset: (96, 6) | essays: 9 | lit: 87


c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(
c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(


,purity,iacs,temperature,time,iacs_final,is_essay
0,99.9,88.260,623.0,30.0,88.350,True
1,99.9,88.260,623.0,60.0,88.655,True
2,99.9,88.260,623.0,90.0,88.800,True
3,99.9,99.825,573.0,30.0,102.340,True
4,99.9,99.390,573.0,30.0,102.230,True


# 3. Base config

In [4]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    presets_a = "best_quality",
    #     presets_a = "medium_quality"
    #     presets_b = "medium_quality"
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

annealing-v1-best-quality__9e081980


# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [5]:
result = runr.run_experiment(df, cfg=base, df_val=df_val)
result["artifacts"]["metrics"]

2026-06-25 12:49:54,798 | INFO | src.modeling.Experiments | === Running annealing-v1-best-quality__9e081980 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       7.47 GB / 31.57 GB (23.7%)
Disk Space Avail:   749.92 GB / 932.08 GB (80.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=5, num_bag_sets=1
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "c:\Users\f

{'rmse': 3.996119956629481,
 'mae': 2.499519852185059,
 'mape': 3.187536363320045,
 'r2': 0.885691000328722}

# 5. Validation set

`df_val` rides along inside the runner: after training + calibration, the
calibrated predictive distribution is evaluated on it and `val_rmse`,
`val_mae`, `val_r2`, `val_cov_*` are appended to the SAME log row — so every
grid run carries fold (OOF) metrics AND validation metrics side by side.

Using `df_schema` as `df_val` is illustrative: those rows are inside the
training data, so val metrics are optimistic. Swap in any held-out
DataFrame with the same columns and nothing else changes.

In [6]:
df_val

,id,purity,initial_diameter,iacs,temperature,time,iacs_final
0,C1_250_30m_T1,99.95,1.2,98.47,523,30,101.08
1,C1_250_30m_T3,99.95,1.2,98.47,523,30,101.08
2,C1_250_60m_T1,99.95,1.2,98.47,523,60,100.82
3,C1_250_60m_T2,99.95,1.2,98.47,523,60,100.82
4,C1_250_90m_T1,99.95,1.2,98.47,523,90,101.22
5,C1_250_90m_T2,99.95,1.2,98.47,523,90,101.22
6,C1_250_30m_AQ_T1,99.95,1.2,98.47,523,30,100.95
7,C1_250_30m_AQ_T2,99.95,1.2,98.47,523,30,100.95
8,C1_300_30m_T1,99.95,1.2,98.47,573,30,102.03
9,C1_300_30m_T2,99.95,1.2,98.47,573,30,102.03


In [7]:
result["artifacts"]["validation_metrics"]

{'rmse': 0.7936130294660075,
 'mae': 0.6655881888544484,
 'mape': 0.6596575497019923,
 'r2': -1.847867144716632,
 'coverage': {0.5: 1.0, 0.8: 1.0, 0.9: 1.0, 0.95: 1.0}}

In [8]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [9]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [10]:
result

{'cfg': ExperimentConfig(process='annealing_iacs', tag='annealing-v1-best-quality', base_tag=None, target='iacs_final', features=('purity', 'iacs', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='best_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=None, y_min=None, fold_seed=42, use_shared_folds=False),
 'run_dir': Path('../../models/annealing_iacs/experiments/annealing-v1-best-quality/annealing-v1-best-quality__9e081980'),
 'artifacts': {'config': {'process': 'annealing_iacs',
   'tag': 'annealing-v1-best-quality',
   'base_tag': None,
   'target': 'iacs_final',
   'features': ['purity', 'iacs', 'temperature', 'time'],
   'weight_on_essay_rows': 1.0,
   'presets_a': 'best_quality',
   'num_bag_folds_a': 5,
   'num_ba

# 6. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [11]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-06-25 12:52:51,140 | INFO | src.modeling.Experiments | Grid: 8 runs over ['time_limit_a', 'num_bag_folds_a', 'weight_on_essay_rows']
2026-06-25 12:52:51,142 | INFO | src.modeling.Experiments | === Running annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=1.0__b2cc9ca8 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       6.55 GB / 31.57 GB (20.8%)
Disk Space Avail:   749.83 GB / 932.08 GB (80.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=10, num_bag_sets=1
Values in column 'weight_col' used as sample weights instead of predictive features. Eva

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,mean_sigma_aleat,n_val,val_rmse,val_mae,val_mape,val_r2,val_cov_0.5,val_cov_0.8,val_cov_0.9,val_cov_0.95
0,annealing-v1-best-quality__9e081980,2026-06-25T12:52:48,174.1,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__9e081980,annealing_iacs,annealing-v1-best-quality,NaN,iacs_final,...,1.19368,36,0.79361,0.66559,0.65966,-1.84787,1.0000,1.0000,1.0,1.0
1,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=1.0__b2cc9ca8,2026-06-25T13:09:05,974.4,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=1.0__b2cc9ca8,annealing_iacs,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=1.0,annealing-v1-best-quality,iacs_final,...,1.18815,36,1.64868,1.26852,1.25711,-11.29061,0.6111,0.9444,1.0,1.0
2,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=2.0__c6973299,2026-06-25T13:32:20,1394.8,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=2.0__c6973299,annealing_iacs,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=2.0,annealing-v1-best-quality,iacs_final,...,1.19206,36,1.69936,1.32685,1.31487,-12.05786,0.6667,0.9444,1.0,1.0
3,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=1.0__b00d83ba,2026-06-25T13:48:19,959.4,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=1.0__b00d83ba,annealing_iacs,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=1.0,annealing-v1-best-quality,iacs_final,...,1.18815,36,1.45122,1.17272,1.16195,-8.52294,0.7222,1.0000,1.0,1.0
4,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=2.0__c2faef57,2026-06-25T14:04:42,982.3,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=2.0__c2faef57,annealing_iacs,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=2.0,annealing-v1-best-quality,iacs_final,...,1.18815,36,0.68192,0.57464,0.56989,-1.10266,0.9444,1.0000,1.0,1.0
5,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=1.0__58129934,2026-06-25T14:30:47,1564.9,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=1.0__58129934,annealing_iacs,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=1.0,annealing-v1-best-quality,iacs_final,...,1.20936,36,1.10730,0.90545,0.89713,-4.54416,0.7778,1.0000,1.0,1.0
6,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=2.0__a960fe25,2026-06-25T14:56:50,1563.5,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v1-best-quality\annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=2.0__a960fe25,annealing_iacs,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=2.0,annealing-v1-best-quality,iacs_final,...,1.20591,36,1.24997,0.97162,0.96267,-6.06481,0.7778,1.0000,1.0,1.0
7,annealing-v1-best-quality__time_lim

In [12]:
# import pickle
# from pathlib import Path
# import pandas as pd

# exp_dir = Path("../../models/annealing_iacs/experiments")
# rows = []
# for run_path in exp_dir.iterdir():
#     pkl = run_path / "artifacts.pkl"
#     if not pkl.exists():
#         continue
#     with open(pkl, "rb") as f:
#         art = pickle.load(f)
#     row = {"run_id": run_path.name}
#     row.update({f"cfg_{k}": v for k, v in art["config"].items()})
#     m = dict(art["metrics"]); m.pop("coverage", None)
#     row.update({k: v for k, v in m.items()})
#     cov = art["metrics"].get("coverage", {})
#     row.update({f"cov_{a}": c for a, c in cov.items()})
#     row["c_opt"] = art["recalibration_c"]
#     if art.get("validation_metrics"):
#         vm = dict(art["validation_metrics"]); vcov = vm.pop("coverage", {})
#         row.update({f"val_{k}": v for k, v in vm.items()})
#         row.update({f"val_cov_{a}": c for a, c in vcov.items()})
#     row["cfg_features"] = "|".join(art["features"])
#     rows.append(row)

# pd.DataFrame(rows).to_csv(exp_dir / "experiments_log.csv", index=False)
# print(f"Rebuilt log with {len(rows)} runs")

# 7. Inspect results

In [13]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_weight_on_essay_rows,cfg_features,rmse,mae,cov_0.9,val_rmse,val_mae,val_cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
7,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=20__weight_on_essay_rows=1.0__d91e7f29,1500,1.0,purity|iacs|temperature|time,2.46436,1.60750,0.8958,1.83345,1.46938,1.0,1.0902,45.83,1.20462,1.19315,2055.6
8,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=20__weight_on_essay_rows=2.0__09e4459d,1500,2.0,purity|iacs|temperature|time,2.48332,1.64380,0.8958,1.82811,1.45586,1.0,1.1735,45.83,1.21766,1.20141,2045.3
4,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=2.0__c2faef57,900,2.0,purity|iacs|temperature|time,2.72366,1.82545,0.8958,0.68192,0.57464,1.0,1.4165,46.88,1.43081,1.18815,982.3
5,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=1.0__58129934,1500,1.0,purity|iacs|temperature|time,2.87629,1.90605,0.8854,1.10730,0.90545,1.0,1.4549,42.71,1.22487,1.20936,1564.9
6,annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=10__weight_on_essay_rows=2.0__a960fe25,1500,2.0,purity|iacs|temperature|time,2.88314,1.93602,0.8958,1.24997,0.97162,1.0,1.6562,40.62,1.17776,1.20591,1563.5
3,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=20__weight_on_essay_rows=1.0__b00d83ba,900,1.0,purity|iacs|temperature|time,2.94994,1.80882,0.9062,1.45122,1.17272,1.0,1.3767,59.38,1.37775,1.18815,959.4
2,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=2.0__c6973299,900,2.0,purity|iacs|temperature|time,3.03089,2.01250,0.8958,1.69936,1.32685,1.0,1.2695,55.21,1.70622,1.19206,1394.8
1,annealing-v1-best-quality__time_limit_a=900__num_bag_folds_a=10__weight_on_essay_rows=1.0__b2cc9ca8,900,1.0,purity|iacs|temperature|time,3.03709,1.99517,0.8854,1.64868,1.26852,1.0,1.2320,46.88,1.55804,1.18815,974.4
0,annealing-v1-best-quality__9e081980,120,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,0.79361,0.66559,1.0,1.4249,47.92,1.67132,1.19368,174.1


In [14]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                 mae           cov_0.9          
                      mean       std      mean       std    mean       std
cfg_time_limit_a                                                          
120               3.996120       NaN  2.499520       NaN  0.9062       NaN
900               2.935395  0.146634  1.910485  0.108236  0.8958  0.008492
1500              2.676778  0.234477  1.773343  0.171620  0.8932  0.005200

# 8. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [15]:
# log already carries the resolved absolute path, so use it directly
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=20__weight_on_essay_rows=1.0__d91e7f29


,alpha,empirical_coverage,gap
0,0.50,0.625000,0.125000
1,0.80,0.812500,0.012500
2,0.90,0.895833,-0.004167
3,0.95,0.916667,-0.033333


In [19]:
run_dir

Path('C:/Users/fmfoa/Projects/uncertainty-aware-predictors/models/annealing_iacs/experiments/annealing-v1-best-quality/annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=20__weight_on_essay_rows=1.0__d91e7f29')

In [17]:
run_dir

Path('C:/Users/fmfoa/Projects/uncertainty-aware-predictors/models/annealing_iacs/experiments/annealing-v1-best-quality/annealing-v1-best-quality__time_limit_a=1500__num_bag_folds_a=20__weight_on_essay_rows=1.0__d91e7f29')